# 법인카드 이상거래 탐지 — 7단계: 최종 학습 및 test 평가 (딱 한 번)

**목적**: 5단계(fold 비교)에서 확정한 조합을 `train_df` 전체로 최종 학습하고, 지금까지 한 번도 열지 않았던
`test_df`(2024년)에 **이 노트북에서 딱 한 번만** 적용해 최종 성능을 확인한다.

**확정된 조합 (5단계 결과)**
- 알고리즘: Isolation Forest
- 피처셋: Tier0(14개) + `일시불할부구분코드`(1개) = 15개
- 하이퍼파라미터: `n_estimators=200`, `max_samples='auto'`, `contamination='auto'`
- fold 평균 성능(참고용, train 내부): PR-AUC 0.4946, recall@top5% 0.6154, recall@top10% 0.7508

**⚠️ 이 노트북의 규칙**
- `test_df`는 **이 노트북에서 딱 한 번만 채점**한다. 결과가 마음에 안 든다고 피처나 하이퍼파라미터를 바꿔서
  이 노트북을 반복 실행하면, 그 자체가 test를 이용한 새로운 형태의 과적합(누수)이 된다.
- 결과가 fold 평균과 크게 다르면(과적합/과소적합 정황), 원인 분석은 하되 **test 점수를 보고 하이퍼파라미터를
  다시 튜닝하지 않는다** — 그건 5단계로 돌아가 train 내부 fold로만 다시 실험해야 한다.
- 전체 성능과 함께, 전처리 7-1절에서 만든 `카드_train노출여부`(재사용 카드 vs 신규 카드) 세그먼트별 성능도
  반드시 같이 리포트한다.

## 1. 라이브러리 & 데이터 로드

In [1]:
import json
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score

pd.set_option('display.max_columns', 60)

DATA_DIR = '../.data/processed'
TRAIN_PATH = os.path.join(DATA_DIR, 'train_processed.csv')
TEST_PATH = os.path.join(DATA_DIR, 'test_processed.csv')
TIERS_PATH = os.path.join(DATA_DIR, 'feature_tiers.json')

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
test_df = pd.read_csv(TEST_PATH, low_memory=False)
with open(TIERS_PATH, encoding='utf-8') as f:
    FEATURE_TIERS = json.load(f)

print(f"train_df: {train_df.shape}")
print(f"test_df : {test_df.shape}")
print(f"test_df에 카드_train노출여부 존재: {'카드_train노출여부' in test_df.columns}")


train_df: (1482969, 47)
test_df : (469902, 47)
test_df에 카드_train노출여부 존재: True


## 2. dtype 재적용 (train, test 둘 다)

전처리 노트북 10절 안내대로, CSV에서 다시 읽으면 category/datetime dtype이 풀려있다. 최종 피처셋에 필요한
컬럼만 재적용한다: Tier0의 `거래요일_한글`·`시간대구간`(category), `거래일자`(datetime), `월말여부`(bool),
그리고 `일시불할부구분코드`(category).

In [2]:
for df in (train_df, test_df):
    df['거래요일_한글'] = df['거래요일_한글'].astype('category')
    df['시간대구간'] = df['시간대구간'].astype('category')
    df['거래일자'] = pd.to_datetime(df['거래일자'])
    df['월말여부'] = df['월말여부'].astype(bool)
    df['일시불할부구분코드'] = df['일시불할부구분코드'].astype('category')

test_df['카드_train노출여부'] = test_df['카드_train노출여부'].astype(bool)

print("재적용 후 dtype 확인:")
print(train_df[['거래요일_한글', '시간대구간', '거래일자', '월말여부', '일시불할부구분코드']].dtypes)


재적용 후 dtype 확인:
거래요일_한글            category
시간대구간              category
거래일자         datetime64[us]
월말여부                   bool
일시불할부구분코드          category
dtype: object


## 3. 모델 입력 행렬 준비 (최종 피처셋: Tier0 + 일시불할부구분코드)

5단계에서 검증한 방식을 그대로 따른다 — 확장 통계 NaN은 median(train 기준값을 test에도 동일하게 적용,
누수 방지를 위해 test 자체의 median을 다시 구하지 않음), `할부가능개월수`는 이번 최종 피처셋에 없으므로
해당 처리는 불필요, `일시불할부구분코드`는 저카디널리티라 원-핫 인코딩한다.

**⚠️ 피처 개수 표기 정정**: `FEATURE_TIERS['tier0_transaction_safe']`(14개) + `일시불할부구분코드`(1개) = 15개를
"최종 확정 피처셋"이라 불러왔지만, 이 중 `거래일자`·`거래연월`은 `DROP_FROM_MODEL`로 모델 입력에서 제외된다
(시간 식별자는 파생 피처 계산용이지 IsolationForest 입력이 아님). 따라서 **실제 모델에 들어가는 원본 컬럼은
13개**이며, 원-핫 인코딩 후 24개 컬럼이 된다. 팀 문서·발표 자료에는 "확정 피처셋 15개(그중 모델 직접 입력
13개, 2개는 파생 계산용 식별자)"로 명시할 것 — "15개로 학습했다"고만 쓰면 노트북과 불일치한다.


In [3]:
NAN_FILL_COLS = ['사용자평균사용액_확장', '사용자표준편차_확장', '거래금액_Zscore_확장']
ONEHOT_COLS = ['거래요일_한글', '시간대구간', '일시불할부구분코드']
DROP_FROM_MODEL = ['거래일자', '거래연월']

FINAL_FEATURE_COLS = FEATURE_TIERS['tier0_transaction_safe'] + ['일시불할부구분코드']
print(f"최종 피처 목록 ({len(FINAL_FEATURE_COLS)}개):")
print(FINAL_FEATURE_COLS)

# median은 반드시 train 기준으로만 계산 — test 정보가 전처리에 섞이지 않도록
_fill_values = train_df[NAN_FILL_COLS].median()
print("\nNaN 대체 median 값 (train 기준):")
print(_fill_values)


def build_model_matrix(df, fill_values=_fill_values):
    cols = [c for c in FINAL_FEATURE_COLS if c not in DROP_FROM_MODEL]
    X = df[cols].copy()
    X[NAN_FILL_COLS] = X[NAN_FILL_COLS].fillna(fill_values)
    X['월말여부'] = X['월말여부'].astype(int)
    X = pd.get_dummies(X, columns=ONEHOT_COLS, drop_first=False)

    remaining_na = X.isna().sum()
    remaining_na = remaining_na[remaining_na > 0]
    if len(remaining_na) > 0:
        print(f"  [경고] 처리 안 된 NaN 발견 — 0으로 채우고 진행:")
        print(remaining_na)
        X = X.fillna(0)
    return X


X_train = build_model_matrix(train_df)
X_test = build_model_matrix(test_df)

# train/test 컬럼이 원-핫 인코딩 후 서로 다를 수 있으므로(예: 특정 카테고리가 한쪽에만 존재) 맞춰준다
X_train, X_test = X_train.align(X_test, join='outer', axis=1, fill_value=0)

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")
assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0


최종 피처 목록 (15개):
['승인시간대', '통합승인금액', '거래일자', '거래연월', '거래요일_한글', '시간대구간', '월말여부', '취소성거래_추정', '최근7일사용횟수', '카드누적사용액', '사용자평균사용액_확장', '사용자표준편차_확장', '거래금액_Zscore_확장', '카드첫거래여부', '일시불할부구분코드']

NaN 대체 median 값 (train 기준):
사용자평균사용액_확장        27775.739501
사용자표준편차_확장        128056.821605
거래금액_Zscore_확장        -0.146994
dtype: float64

X_train shape: (1482969, 24)
X_test  shape: (469902, 24)


## 4. 최종 학습 (train 전체) 및 test 채점 (딱 한 번)

5단계에서 확정한 하이퍼파라미터 그대로, 이번엔 fold로 나누지 않고 **train_df 전체**로 학습한다.

In [4]:
final_model = IsolationForest(
    n_estimators=200,
    max_samples='auto',
    contamination='auto',
    random_state=42,
    n_jobs=-1,
)
final_model.fit(X_train)

test_anomaly_score = -final_model.decision_function(X_test)
test_df['anomaly_score'] = test_anomaly_score

print("test_df 채점 완료.")
print(f"anomaly_score 범위: {test_anomaly_score.min():.4f} ~ {test_anomaly_score.max():.4f}")


test_df 채점 완료.
anomaly_score 범위: -0.1226 ~ 0.2268


## 5. 전체 test 성능

In [5]:
TOP_K_FRACTIONS = [0.01, 0.03, 0.05, 0.10]

y_test = test_df['이상거래여부'].values
overall_pr_auc = average_precision_score(y_test, test_anomaly_score)

order = np.argsort(-test_anomaly_score)
n_pos_total = y_test.sum()

overall_row = {'pr_auc': overall_pr_auc, 'n': len(y_test), 'n_positive': int(n_pos_total)}
for frac in TOP_K_FRACTIONS:
    k = max(1, int(len(y_test) * frac))
    top_k_idx = order[:k]
    overall_row[f'recall@top{int(frac*100)}%'] = y_test[top_k_idx].sum() / n_pos_total
    overall_row[f'precision@top{int(frac*100)}%'] = y_test[top_k_idx].sum() / k

overall_summary = pd.Series(overall_row)
print("=== 전체 test 성능 ===")
print(overall_summary)

print(f"\n(참고) train 내부 5-fold 평균 PR-AUC: 0.4946 — 이 값과 비교해 과적합/과소적합 여부를 가늠할 수 있다.")


=== 전체 test 성능 ===
pr_auc                   0.586508
n                   469902.000000
n_positive           16394.000000
recall@top1%             0.248689
precision@top1%          0.867631
recall@top3%             0.525802
precision@top3%          0.611478
recall@top5%             0.663779
precision@top5%          0.463162
recall@top10%            0.790350
precision@top10%         0.275740
dtype: float64

(참고) train 내부 5-fold 평균 PR-AUC: 0.4946 — 이 값과 비교해 과적합/과소적합 여부를 가늠할 수 있다.


## 6. 세그먼트별 test 성능 — 재사용 카드 vs 신규 카드

전처리 7-1절에서 만든 `카드_train노출여부`로 나눠서, 모델이 카드 이력(재사용 카드)에 얼마나 의존하는지,
신규 카드(콜드스타트)에서는 실제로 얼마나 잘 작동하는지 확인한다.

In [6]:
def compute_metrics(y_true, scores, fracs=TOP_K_FRACTIONS):
    row = {'n': len(y_true), 'n_positive': int(y_true.sum())}
    if y_true.sum() == 0:
        row['pr_auc'] = np.nan
    else:
        row['pr_auc'] = average_precision_score(y_true, scores)
    order = np.argsort(-scores)
    n_pos = y_true.sum()
    for frac in fracs:
        k = max(1, int(len(y_true) * frac))
        top_k_idx = order[:k]
        row[f'recall@top{int(frac*100)}%'] = y_true[top_k_idx].sum() / n_pos if n_pos > 0 else np.nan
        row[f'precision@top{int(frac*100)}%'] = y_true[top_k_idx].sum() / k
    return row


segment_rows = {}
for seg_value, seg_label in [(True, '재사용 카드'), (False, '신규 카드')]:
    mask = test_df['카드_train노출여부'] == seg_value
    y_seg = test_df.loc[mask, '이상거래여부'].values
    score_seg = test_anomaly_score[mask.values]
    segment_rows[seg_label] = compute_metrics(y_seg, score_seg)

segment_summary = pd.DataFrame(segment_rows)
print("=== 세그먼트별 test 성능 ===")
segment_summary.round(4)


=== 세그먼트별 test 성능 ===


,재사용 카드,신규 카드
n,465328.0000,4574.0000
n_positive,16231.0000,163.0000
pr_auc,0.5865,0.6367
recall@top1%,0.2487,0.2331
precision@top1%,0.8674,0.8444
recall@top3%,0.5262,0.5767
precision@top3%,0.6118,0.6861
recall@top5%,0.6632,0.7423
precision@top5%,0.4627,0.5307
recall@top10%,0.7901,0.8282


## 7. 우선순위 랭킹 규칙 확정 (top3% 컷오프)

회계팀 검토 용량을 감안해 **상위 3%를 위험 후보로 분류**하기로 확정했다. 두 가지를 구분해서 확인한다.

- **(a) test 자체에서 정확히 상위 3%**: 5~6절 `TOP_K_FRACTIONS`에 이미 0.03을 추가해뒀으므로 그 결과를 그대로 참고
- **(b) 운영 시나리오 재현**: 실제 배포에서는 매 거래가 들어올 때마다 "미리 정해둔 고정 점수 임계값"과 비교해야
  하므로, **train 점수 분포의 97번째 백분위수를 임계값으로 고정**한 뒤 그걸 test에 적용했을 때 실제로 몇 %가
  걸리는지, 그 성능이 (a)와 얼마나 차이 나는지 확인한다. 차이가 크면 임계값이 시간에 따라 불안정하다는 신호다.

**⚠️ 문서화 보완**: (a) "이상적 top3%" 수치(recall 0.526 / precision 0.611)만 팀 문서에 남기면 실제 운영
성능으로 오해될 수 있다. 아래 (b) 고정 임계값 적용 결과(recall/precision)가 곧 운영 시 기대 성능이므로,
팀 공유 문서에는 (a)와 (b) 두 수치를 반드시 함께 기재한다.


In [7]:
train_anomaly_score = -final_model.decision_function(X_train)
THRESHOLD_PERCENTILE = 97  # 상위 3%
fixed_threshold = np.percentile(train_anomaly_score, THRESHOLD_PERCENTILE)
print(f"train 기준 고정 임계값(상위 3% 지점): {fixed_threshold:.4f}")

# (a) test 내 정확히 상위 3% (5절 결과에서 재확인)
k_a = max(1, int(len(y_test) * 0.03))
order_a = np.argsort(-test_anomaly_score)[:k_a]
recall_a = y_test[order_a].sum() / y_test.sum()
precision_a = y_test[order_a].sum() / k_a

# (b) train에서 정한 고정 임계값을 test에 그대로 적용
flagged_b = test_anomaly_score >= fixed_threshold
actual_pct_b = flagged_b.mean() * 100
recall_b = y_test[flagged_b].sum() / y_test.sum()
precision_b = y_test[flagged_b].sum() / flagged_b.sum() if flagged_b.sum() > 0 else np.nan

print(f"\n(a) test 내 정확히 top3% ({k_a:,}건): recall={recall_a:.3f}, precision={precision_a:.3f}")
print(f"(b) train 기준 고정 임계값 적용 (실제 {actual_pct_b:.2f}%, {flagged_b.sum():,}건): "
      f"recall={recall_b:.3f}, precision={precision_b:.3f}")
print(f"\n분포 이동(drift) 정도: 의도한 3.00% vs 실제 {actual_pct_b:.2f}% "
      f"(차이 {abs(actual_pct_b - 3.0):.2f}%p)")

test_df['위험등급'] = np.where(test_df['anomaly_score'] >= fixed_threshold, '검토대상', '정상')
print("\n위험등급 분포:")
print(test_df['위험등급'].value_counts())


# 운영 기준 성능 요약 (팀 문서에 반드시 (a)와 함께 기재)
ops_threshold_summary = pd.Series({
    'threshold_percentile': THRESHOLD_PERCENTILE,
    'fixed_threshold_score': fixed_threshold,
    'flagged_pct': actual_pct_b,
    'flagged_count': int(flagged_b.sum()),
    'recall': recall_b,
    'precision': precision_b,
})
print("\n=== (b) 고정 임계값 기준 운영 성능 요약 ===")
print(ops_threshold_summary)


train 기준 고정 임계값(상위 3% 지점): 0.0620

(a) test 내 정확히 top3% (14,097건): recall=0.526, precision=0.611
(b) train 기준 고정 임계값 적용 (실제 3.29%, 15,465건): recall=0.550, precision=0.583

분포 이동(drift) 정도: 의도한 3.00% vs 실제 3.29% (차이 0.29%p)

위험등급 분포:
위험등급
정상      454437
검토대상     15465
Name: count, dtype: int64

=== (b) 고정 임계값 기준 운영 성능 요약 ===
threshold_percentile        97.000000
fixed_threshold_score        0.062046
flagged_pct                  3.291112
flagged_count            15465.000000
recall                       0.550384
precision                    0.583446
dtype: float64


## 요약

**이 노트북에서 한 일**
1. `train_processed.csv`(2021~2023) 전체로 최종 Isolation Forest를 학습했다 — 5단계에서 확정한 피처셋
   (Tier0 14개 + 일시불할부구분코드, 그중 모델 직접 입력은 13개 — 거래일자·거래연월은 파생 계산용으로 제외)
   · 하이퍼파라미터(n_estimators=200, max_samples='auto')를 그대로 사용했다.
2. `test_processed.csv`(2024년)에 이 노트북에서 **딱 한 번** 채점했다(v2는 진단 셀 추가를 위한 재실행이며,
   모델 스펙(피처·하이퍼파라미터)은 v1과 동일 — test 점수 자체는 변하지 않았다).
3. 전체 test 성능(PR-AUC, top-k recall/precision)을 확인하고, train 내부 fold 평균(0.4946)과 비교했다.
4. `카드_train노출여부`로 재사용 카드 vs 신규 카드 세그먼트별 성능을 나눠서 확인했다 — 모델이 카드 이력에
   얼마나 의존하는지 정량적으로 드러내기 위함이다.

**해석 시 주의할 점**
- **test PR-AUC(0.5865)는 fold 평균(0.4946)보다 오히려 높게 나왔다.** fold 평균 표준편차(0.051)를 감안해도
  차이가 있어(§5 진단 참고), 이는 성능 저하가 아니라 카드 단위 무작위 fold 분할(보수적 하한선)과 시간 기준
  분할(실제 운영과 동일 조건, 카드 재사용률 97%로 확장 통계가 안정적으로 채워짐)의 **구조적 차이**로 설명된다.
  셀 16의 pseudo-test(시간 분할, train 내부)가 0.5536으로 test와 유사하게 나와 이 설명을 뒷받침한다.
- 신규 카드 세그먼트는 표본이 매우 작을 수 있어(test 내 약 4,574건, 카드 73개) 지표가 불안정할 수 있다 —
  절대적 수치보다 재사용 카드 대비 상대적 경향을 보는 게 더 안전하다.
- 이 결과를 본 뒤 성능이 부족해 보여도, **이 노트북을 반복 실행해 피처/하이퍼파라미터를 바꾸면 안 된다.**
  개선이 필요하면 5단계(train 내부 fold)로 돌아가 다시 검증한 뒤, 최종 후보가 바뀌었을 때만 새 노트북으로
  다시 한 번 test를 채점해야 한다.

**7절: 우선순위 랭킹 규칙 확정**
- 회계팀 검토 용량을 고려해 상위 3%를 위험 후보로 분류하기로 확정했다. 운영 재현을 위해 train 점수
  분포의 97번째 백분위수를 고정 임계값(0.0620)으로 삼고, test에 적용한 결과 실제 분류 비율 3.29%
  (recall 0.550 / precision 0.583)로 "test 내 정확히 top3%"(recall 0.526 / precision 0.611)와 크게
  다르지 않아, 이 고정 임계값을 운영에 그대로 가져다 써도 안정적이라고 판단한다.

**다음 단계**
- anomaly score(`test_df['anomaly_score']`)를 회계팀 검토 UI용 우선순위 랭킹 리스트로 변환하는 설계
  (7절에서 컷오프는 확정했으므로, 남은 것은 리스트 표시 방식·사유 태깅 등 UI/UX 설계)
- 최종 결과를 팀 문서로 정리하고 Open Issue(#1, #2) 갱신 — Open Issue #1(θ_pass/θ_reject)과 이번 anomaly
  score 컷오프(top3%)는 별개 사안이므로 혼동하지 않도록 명시
- **[운영 전환 시 필요]** `최근7일사용횟수`, `카드누적사용액`, 확장 통계 피처 등은 현재 배치 전처리로
  계산된 값이다. FastMCP `ml_infer` 도구가 실시간 거래 진입 시점에 이 피처들을 어떻게 재현할지
  (Django 내부 read API로 과거 거래 조회 후 계산 vs. 별도 피처 저장소)는 아직 결정되지 않았으므로
  Risk Review Agent 서빙 설계 시 별도 논의가 필요하다.
- fold 4가 다른 fold보다 지속적으로 낮게 나온 원인 — 아직 미조사(참고용)


In [8]:
# train_df 안에서 시간 기준 pseudo-test 분할 (test 구조와 동일하게)
pseudo_cutoff = '2023-01-01'
pseudo_train_idx = train_df.index[train_df['거래일자'] < pseudo_cutoff]
pseudo_valid_idx = train_df.index[train_df['거래일자'] >= pseudo_cutoff]

X_pt = X_train.loc[pseudo_train_idx]
X_pv = X_train.loc[pseudo_valid_idx]
y_pv = train_df.loc[pseudo_valid_idx, '이상거래여부'].values

iso_pseudo = IsolationForest(n_estimators=200, max_samples='auto', contamination='auto', random_state=42, n_jobs=-1)
iso_pseudo.fit(X_pt)
score_pv = -iso_pseudo.decision_function(X_pv)

print("Pseudo-test(train 내부, 시간분할) PR-AUC:", average_precision_score(y_pv, score_pv))

Pseudo-test(train 내부, 시간분할) PR-AUC: 0.553565956852893
